In [ ]:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import sklearn.metrics as metrics
from matplotlib import pyplot as plt

from snowflake.ml.modeling.xgboost import XGBClassifier as XGBC
from snowflake.ml.modeling import metrics as mtrx
import seaborn as sns

In [ ]:
sesssion=get_active_session()

In [ ]:
session

In [ ]:
X,y=make_classification(n_samples=40000, n_features=6, n_informative=4,
                       n_redundant=1, random_state=0, shuffle=True)

In [ ]:
X=pd.DataFrame(data=X,columns="X1 X2 X3 X4 X5 X6".split(" "))
y=pd.DataFrame(data=y, columns=['Y'])

In [ ]:
pdf=pd.concat(objs=[X,y], axis=1)

In [ ]:
pdf

In [ ]:
df=session.create_dataframe(pdf)
df.show()

In [ ]:
df.write.mode("overwrite").save_as_table("test.public.classification_dataset_for_metrics")

## Metrics using Sklearn

In [ ]:
clf=XGBClassifier()

In [ ]:
clf.fit(X_train,y_train)

In [ ]:
train_data_pred=clf.predict(X_train)

In [ ]:
training_accuracy=metrics.accuracy_score(y_train, train_data_pred)

In [ ]:
training_accuracy

In [ ]:
test_data_pred=clf.predict(X_test)
eval_accuracy=metrics.accuracy_score(y_test, test_data_pred)
eval_accuracy

In [ ]:
metrics.precision_score(y_test, test_data_pred)

In [ ]:
metrics.recall_score(y_test, test_data_pred)

In [ ]:
metrics.f1_score(y_test, test_data_pred)

In [ ]:
matrix=metrics.confusion_matrix(y_train, train_data_pred)
matrix

In [ ]:
plt.figure(figsize=(20,20))
sns.heatmap(matrix,annot=True, fmt=".0f",cmap='Blues')
plt.show()

## Metrics using SnowparkML

In [ ]:
df=session.table("test.public.classification_dataset_for_metrics")

In [ ]:
train_data, test_data=df.random_split(weights=[0.6,0.4], seed=0)

In [ ]:
train_data.show()

In [ ]:
clf=XGBC(
    input_cols="X1 X2 X3 X4 X5 X6".split(" "),
    label_cols=['Y'],
    output_cols=['PREDICTIONS']
)

In [ ]:
clf.fit(train_data)

In [ ]:
train_data_pred=clf.predict(train_data)
train_data_pred.select(
    'PREDICTIONS'
).show()

In [ ]:
training_accuracy=mtrx.accuracy_score(
    df=train_data_pred,
    y_true_col_names=['Y'],
    y_pred_col_names=['PREDICTIONS']
)
training_accuracy

In [ ]:
test_data_pred=clf.predict(test_data)
test_data_pred.select(
    'PREDICTIONS'
).show()

In [ ]:
eval_accuracy=mtrx.accuracy_score(
    df=test_data_pred,
    y_true_col_names=['Y'],
    y_pred_col_names=['PREDICTIONS']
)
eval_accuracy

In [ ]:
eval_precision=mtrx.precision_score(
    df=test_data_pred,
    y_true_col_names=['Y'],
    y_pred_col_names=['PREDICTIONS']
)
eval_precision

In [ ]:
eval_recall=mtrx.recall_score(
    df=test_data_pred,
    y_true_col_names=['Y'],
    y_pred_col_names=['PREDICTIONS']
)
eval_recall

In [ ]:
eval_f1_score=mtrx.f1_score(
    df=test_data_pred,
    y_true_col_names=['Y'],
    y_pred_col_names=['PREDICTIONS']
)
eval_f1_score

In [ ]:
mdf=session.create_dataframe(
    data=[
        ("training_accuracy",float(training_accuracy)),
        ("eval_accuracy",float(eval_accuracy)),
        ("precision",float(eval_precision)),
        ("recall", float(eval_recall)),
        ("f1_score", float(eval_f1_score))
    ],
    schema=("metric","value")
)

In [ ]:
mdf.show()